# From dinner request to spice kit

This Moji Masala demonstration matches a cooking request against a dated product-and-recipe catalog. It uses existing [Moji website recipes](https://www.mojimasala.com/blogs/recipes) and [product information](https://www.mojimasala.com/pages/ai-shopping-guide), with source links on each record. No website content is changed.

The first examples run locally without an API key. The last two cells optionally use OpenAI and Claude to extract requirements through the same tool. This is a single-brand demonstration, not an independent comparison of spice brands.


## Setup

From the repository root, follow the [README installation instructions](../README.md), including `python -m pip install --upgrade pip` and `python -m pip install -e '.[providers]'` for the optional provider cells. Select that Python environment as the notebook kernel. Jupyter is a separate optional notebook runner; the command-line examples do not require it.

Keys belong in local environment variables `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`, never in the notebook. The checked-in notebook contains no executed outputs.


In [ ]:
from moji_demo.catalog import load_catalog, validate_catalog
from moji_demo.matching import match_recipes

catalog = load_catalog()
validation = validate_catalog(catalog)
print(validation)


## A grounded match

The local matcher resolves dish aliases and applies explicit constraints. It returns original recipe quantities and source URLs rather than asking a model to invent a recipe. The snapshot's coverage and source limitations are documented in [data/README.md](../data/README.md).


In [ ]:
def summarize(result):
    return {
        "status": result["status"],
        "message": result["message"],
        "catalog_verified_at": result["catalog_verified_at"],
        "matches": [
            {
                "product": item["product"]["name"],
                "product_url": item["product"]["product_url"],
                "recipe": item["recipe"]["title"],
                "recipe_url": item["recipe"]["url"],
                "servings": item["recipe"]["servings"],
                "total_minutes": item["recipe"]["total_minutes"],
                "ingredients": item["recipe"]["ingredients"],
                "source_urls": item["source_urls"],
            }
            for item in result["matches"]
        ],
    }

dal = match_recipes(dish="lentil dal", vegan=True, catalog=catalog)
assert dal["status"] == "matched"
summarize(dal)


## A blend cannot replace one measured spice

A dish-specific packet is intended for its linked recipe. An individual-spice request should not trigger a blended-product substitution.


In [ ]:
individual_spice = match_recipes(
    dish="one teaspoon of cumin", single_spice=True, catalog=catalog
)
assert individual_spice["status"] == "no_match"
summarize(individual_spice)


## Check the finished recipe, not just the blend

A vegan spice blend does not make a chicken recipe vegan. The matcher checks recipe diet metadata separately and does not invent a tofu adaptation. It also declines strict time constraints when the source does not verify them.


In [ ]:
vegan_chicken = match_recipes(
    dish="tandoori chicken", vegan=True, catalog=catalog
)
assert vegan_chicken["status"] == "no_match"
summarize(vegan_chicken)


## Optional live provider examples

Keep `RUN_LIVE = False` for a key-free walkthrough. Change it deliberately only when you have configured the relevant API key and are ready for billed provider calls. Each provider cell checks its key before importing and calling the SDK integration.

Both integrations call the same matching tool and preserve the provider's tool-call continuation state. Final model wording still needs review. See [provider implementation notes](../examples/README.md) and the [verification record](../docs/verification.md).


In [ ]:
import os

RUN_LIVE = False
PROMPT = (
    "I have lentils, onions and tomatoes. Which Moji blend and recipe "
    "fit a vegan dinner? Include the source recipe and product links."
)


In [ ]:
# OpenAI: skipped unless explicitly enabled above.
if RUN_LIVE:
    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError("Set OPENAI_API_KEY locally before enabling this cell.")
    from moji_demo.providers import run_openai

    openai_answer = run_openai(PROMPT)
    print(openai_answer)
else:
    print("OpenAI call skipped: RUN_LIVE is False.")


In [ ]:
# Claude: skipped unless explicitly enabled above.
if RUN_LIVE:
    if not os.environ.get("ANTHROPIC_API_KEY", "").strip():
        raise RuntimeError("Set ANTHROPIC_API_KEY locally before enabling this cell.")
    from moji_demo.providers import run_claude

    claude_answer = run_claude(PROMPT)
    print(claude_answer)
else:
    print("Claude call skipped: RUN_LIVE is False.")


## Distribution and next steps

This notebook is part of the Moji-owned project. An official OpenAI or Claude cookbook proposal is a separate submission requiring maintainer review; none is implied by this example. Publishing this notebook does not train a model on demand or ensure that consumer ChatGPT or Claude recommends Moji.

Continue with the [project README](../README.md), [five-step rollout plan](../docs/rollout-plan.md), [measurement worksheet](../docs/measurement.md), and [proposal drafts](../docs/cookbook-proposals.md). Source recipe content retains its existing rights; see [data provenance and rights](../data/README.md).
